# 06 — Decoding Enhancements in Google Colab (Flickr8k CNN–LSTM)

This notebook provides a runnable companion for **PR #7** (`src/decoding_enhancements.py`). It tests the *trained* caption decoder using four inference strategies:

| Strategy | Function and options | Purpose |
|---|---|---|
| Greedy baseline | `greedy_decode(...)` | Pick the most probable token each step |
| Standard beam baseline | `beam_search_decode(..., beam_width=3, alpha=0)` | Keep 3 candidate sequences |
| **Top-k sampling** | `greedy_decode(..., top_k=3, rng=...)` | Sample from 3 most probable words; seeded |
| **Length-normalised beam** | `beam_search_decode(..., beam_width=3, alpha=0.7)` | Counter some preference for short sequences |

All strategies use **the same frozen VGG16 features, LSTM checkpoint, tokenizer, and image splits**. VGG16/LSTM **are not retrained**.

**Important:** No precomputed BLEU values or example captions are claimed here. The notebook generates fresh outputs from *your saved model*. Tuning should use **validation images**, and the held-out **test set** should be used for a final comparison only after parameters are fixed.

## 1. Settings — change these to match your Google Drive

This branch starts from PR #7 and includes both `src/decoding_enhancements.py` and `src/caption_decoding.py`.

**If your trained checkpoint is not under `MyDrive/NIT3251_captioning/runs/lstm_run1/`**, set `MODEL_DIR_OVERRIDE` to the *folder* containing `caption_model.keras` (or `best_caption_model.keras`) and three JSON files. Features normally reside in `MyDrive/NIT3251_captioning/features/`.

Use `EXAMPLE_SPLIT="val"` when deciding the beam width, top-k or alpha. Save the 810-image test set for final reporting.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/JustinDang69/Automatic-Image-Captioning-with-CNN-and-LSTM-.git"
REPO_BRANCH = "feature/colab-decoding-enhancements"
DRIVE_PROJECT = Path("/content/drive/MyDrive/NIT3251_captioning")
RUN_NAME = "lstm_run1"

MODEL_DIR_OVERRIDE = None     # e.g. "/content/drive/MyDrive/MyModelFolder"
FEATURES_DIR_OVERRIDE = None  # e.g. "/content/drive/MyDrive/VGG16_features"
DATA_DIR_OVERRIDE = None      # e.g. "/content/drive/MyDrive/Flickr8k"

BEAM_WIDTH = 3
TOP_K = 3
ALPHA = 0.7
RANDOM_SEED = 42
EXAMPLE_SPLIT = "val"          # "val" for tuning; "test" for final qualitative examples
NUM_EXAMPLES = 3
VALIDATION_IMAGE_LIMIT = 20   # exploratory only

RUN_EXPLORATORY_BLEU = False
RUN_FINAL_TEST_BLEU = False  # True only after hyperparameters are fixed

assert BEAM_WIDTH >= 1 and TOP_K >= 1
assert ALPHA >= 0
assert EXAMPLE_SPLIT in {"val", "test"}

## 2. Mount Drive and download the right GitHub branch

The folder `/content/nit3251-enhancements` is deliberately separate from the existing `/content/nit3251-captioning` clone. This avoids accidentally importing older `main` versions of the decoding functions.

If you previously imported a different version of `src.caption_decoding` in the same session, use **Runtime → Restart session**, then run this notebook in order.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/nit3251-enhancements")
    if not (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        for cmd in (
            ["fetch", "origin"],
            ["checkout", REPO_BRANCH],
            ["pull", "--ff-only"],
        ):
            subprocess.run(["git", "-C", str(PROJECT_ROOT), *cmd], check=True)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory:", Path.cwd())
print("Git branch:", subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip())

## 3. Install packages and run PR #7's unit tests

Dependencies are already declared in `requirements.txt`, including TensorFlow/Keras, NLTK, Matplotlib, pandas, Pillow and KaggleHub.

The included tests check the standalone **length-normalisation** and **top-k sampling** functions with synthetic inputs. Passing tests does **not** by itself establish image-caption quality.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_decoding.py"],
    check=True,
)
print("Dependency installation and decoding-enhancement unit tests finished.")

## 4. Confirm the actual imported implementation

The enhancements in PR #7 integrate with `src.caption_decoding`; the notebook must import **these existing function names**. In particular, `load_artifacts`, `generate_caption` and `beam_search_caption` are **not** provided by that PR.

- `top_k_sampling` and `length_normalize_score`: `src.decoding_enhancements`
- `load_assets`, `greedy_decode`, `beam_search_decode`: `src.caption_decoding`
- `_reference_tokens`: matches the repository's existing caption cleaning.

In [ ]:
import json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from nltk.translate.bleu_score import corpus_bleu

from src.decoding_enhancements import (
    top_k_sampling, length_normalize_score,
)
from src.caption_decoding import (
    load_assets, greedy_decode, beam_search_decode, _reference_tokens,
)
from src.data_preprocessing import get_dataset_paths

print("Length-normalized score example:", length_normalize_score(-5.0, 8, alpha=ALPHA))
print("Reproducible top-k example:", top_k_sampling(
    np.array([0.6, 0.3, 0.1]), top_k=2, rng=np.random.default_rng(42)
))

## 5. Locate checkpoint, vocabulary and VGG16 features

**Required checkpoint folder contents**: `caption_model.keras` (or `best_caption_model.keras`), `tokenizer.json`, `splits.json`, and `metadata.json`.

**Required feature folder contents**: `features.npy`, `image_ids.json`, `progress.npy`, and `metadata.json`.

Your model weights and feature arrays are **not stored on GitHub**. If the files were generated by a teammate, copy or add a Drive shortcut to the right folder and adjust the overrides above. This cell gives actionable errors instead of attempting inference with missing files.

In [ ]:
MODEL_DIR = (Path(MODEL_DIR_OVERRIDE) if MODEL_DIR_OVERRIDE
             else DRIVE_PROJECT / "runs" / RUN_NAME)
FEATURES_DIR = (Path(FEATURES_DIR_OVERRIDE) if FEATURES_DIR_OVERRIDE
                else DRIVE_PROJECT / "features")
OUTPUT_DIR = DRIVE_PROJECT / "runs" / RUN_NAME / "decoding_enhancements"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_names = ["caption_model.keras", "best_caption_model.keras"]
checkpoint_found = next(
    (MODEL_DIR / f for f in checkpoint_names if (MODEL_DIR / f).is_file()), None
)
required_model_json = ["tokenizer.json", "splits.json", "metadata.json"]
required_features = ["features.npy", "image_ids.json", "progress.npy", "metadata.json"]
model_missing = [f for f in required_model_json if not (MODEL_DIR / f).is_file()]
features_missing = [f for f in required_features if not (FEATURES_DIR / f).is_file()]

print("MODEL_DIR:", MODEL_DIR)
print("Checkpoint found:", checkpoint_found)
print("Missing model metadata:", model_missing)
print("FEATURES_DIR:", FEATURES_DIR)
print("Missing features:", features_missing)

if checkpoint_found is None or model_missing:
    nearby = sorted(str(x) for x in DRIVE_PROJECT.glob("runs/*/*.keras")) if DRIVE_PROJECT.exists() else []
    raise FileNotFoundError(
        "Trained model or metadata is missing. Check MODEL_DIR_OVERRIDE / RUN_NAME. "
        "Nearby checkpoints: " + repr(nearby[:20]) +
        ". A trained .keras checkpoint cannot be recreated from GitHub code."
    )
if features_missing:
    raise FileNotFoundError(
        "Missing pre-extracted VGG16 files in FEATURES_DIR: " + repr(features_missing)
    )

## 6. Locate Flickr8k images and reference captions

If Flickr8k is already in Drive, the notebook uses it. Otherwise Colab downloads it through KaggleHub. Only **image files and captions** are downloaded; the pretrained LSTM checkpoint and saved VGG16 features must exist separately.

The same image IDs and saved test split from the training run are retained.

In [ ]:
if DATA_DIR_OVERRIDE:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)
else:
    drive_data = DRIVE_PROJECT / "flickr8k"
    if (drive_data / "Images").is_dir() and (drive_data / "captions.txt").is_file():
        DATA_DIR = drive_data
    else:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download("adityajn105/flickr8k"))
        candidates = [downloaded, *downloaded.iterdir()]
        DATA_DIR = next(
            (d for d in candidates
             if d.is_dir() and (d / "Images").is_dir()
             and (d / "captions.txt").is_file()),
            None,
        )
        if DATA_DIR is None:
            raise FileNotFoundError(
                f"Downloaded Flickr8k but could not find Images/ and captions.txt under {downloaded}"
            )
images_dir, captions_path = get_dataset_paths(DATA_DIR)
print("Flickr8k:", DATA_DIR, "| Images:", images_dir, "| Captions:", captions_path)

## 7. Load the **same** saved CNN–LSTM model used by the baselines

`load_assets` loads the checkpoint without compiling for training, reconstructs the exact tokenizer and image split, memory-maps the 4096-feature vectors, and loads all human-written captions.

The model predicts a distribution over the same vocabulary the training process used. No CNN or LSTM weights are updated during inference.

In [ ]:
assets = load_assets(
    model_dir=MODEL_DIR,
    features_dir=FEATURES_DIR,
    data_dir=DATA_DIR,
)
print("Model inputs:", assets.model.input_shape)
print("Model output:", assets.model.output_shape)
print("Max caption length:", assets.max_length)
print("Vocabulary entries:", assets.tokenizer.vocab_size)
print("Split sizes:", {name: len(ids) for name, ids in assets.splits.items()})
assert assets.model.output_shape[-1] == assets.tokenizer.vocab_size
assert len(assets.splits["test"]) > 0

## 8. Compare the four strategies on the **same** image

The method below is only a small adapter around PR #7's existing functions; it does not reimplement the search algorithms.

- `greedy`: deterministic baseline
- `beam`: standard beam with `alpha=0`
- `top_k`: stochastic sampling with a seeded random generator
- `length_norm`: beam search with `alpha=0.7`

The score normalisation formula in PR #7 is `raw_log_probability / sequence_length**alpha`. It alters **candidate ranking**, not the model weights. The top-k value for sampling is distinct from the **beam width** of beam search.

In [ ]:
METHODS = ("greedy", "beam", "top_k", "length_norm")

def generate_all_methods(image_id, rng):
    if image_id not in assets.row_of:
        raise KeyError(f"VGG16 image feature not found: {image_id}")
    feature = assets.features[assets.row_of[image_id]]
    arguments = (assets.model, feature, assets.tokenizer, assets.max_length)
    predictions = {}
    timings = {}

    for method in METHODS:
        started = time.perf_counter()
        if method == "greedy":
            caption = greedy_decode(*arguments)
        elif method == "beam":
            caption = beam_search_decode(*arguments, beam_width=BEAM_WIDTH, alpha=0.0)
        elif method == "top_k":
            caption = greedy_decode(*arguments, top_k=TOP_K, rng=rng)
        else:
            caption = beam_search_decode(*arguments, beam_width=BEAM_WIDTH, alpha=ALPHA)
        predictions[method] = caption
        timings[method] = time.perf_counter() - started

    return {
        "image_id": image_id,
        "references": [" ".join(_reference_tokens(c))
                       for c in assets.references[image_id]],
        "captions": predictions,
        "seconds": timings,
    }

## 9. Generate validation examples for qualitative error analysis

Validation examples are useful for choosing decoding parameters without repeatedly optimising against the final test set. `RANDOM_SEED` ensures a reproducible image selection and top-k sampling sequence.

Compare incorrect objects, missing people, incorrect activities, overly generic descriptions and unfinished captions. Larger beam widths are not guaranteed to improve accuracy.

In [ ]:
selected_ids = random.Random(RANDOM_SEED).sample(
    assets.splits[EXAMPLE_SPLIT],
    k=min(NUM_EXAMPLES, len(assets.splits[EXAMPLE_SPLIT])),
)
sampling_rng = np.random.default_rng(RANDOM_SEED)
examples = [generate_all_methods(img_id, sampling_rng) for img_id in selected_ids]

rows = []
for item in examples:
    print("\nIMAGE:", item["image_id"])
    print("Reference:", item["references"][0])
    for method in METHODS:
        print(f"{method:12s}: {item['captions'][method]}")
    rows.append({
        "Image": item["image_id"],
        "Reference": item["references"][0],
        "Greedy": item["captions"]["greedy"],
        "Beam": item["captions"]["beam"],
        f"Top-k (k={TOP_K})": item["captions"]["top_k"],
        f"Length norm (alpha={ALPHA})": item["captions"]["length_norm"],
    })

display(pd.DataFrame(rows))

## 10. Display the **actual images** and save the caption comparisons

The figure displays each Flickr8k photo alongside its five human-written reference captions and the four model-generated captions. It is saved to your Google Drive for the **Results and Error Analysis** sections.

Image display requires the Flickr8k `Images/` directory. The model and feature extraction are not run again.

In [ ]:
fig, axes = plt.subplots(len(examples), 2,
                          figsize=(16, max(4.5 * len(examples), 5)),
                          squeeze=False,
                          gridspec_kw={"width_ratios": [1, 1.7]})
for (left, right), item in zip(axes, examples):
    with Image.open(assets.images_dir / item["image_id"]) as img:
        left.imshow(img.convert("RGB"))
    left.set_title(item["image_id"], fontsize=10)
    left.axis("off")

    import textwrap
    lines = ["Human references:"]
    lines += [f"{i+1}. {caption}" for i, caption in enumerate(item["references"])]
    lines += [
        "",
        f"Greedy: {item['captions']['greedy']}",
        f"Standard beam (w={BEAM_WIDTH}): {item['captions']['beam']}",
        f"Top-k (k={TOP_K}): {item['captions']['top_k']}",
        f"Length-normalised (alpha={ALPHA}): {item['captions']['length_norm']}",
    ]
    wrapped = [textwrap.fill(x, width=79, subsequent_indent="    ") for x in lines]
    right.text(0.0, 0.99, "\n".join(wrapped), va="top",
               transform=right.transAxes, fontsize=9, linespacing=1.35)
    right.axis("off")

fig.tight_layout(h_pad=2)
figure_path = OUTPUT_DIR / f"examples_{EXAMPLE_SPLIT}_beam{BEAM_WIDTH}_topk{TOP_K}_alpha{ALPHA}.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

json_path = OUTPUT_DIR / f"examples_{EXAMPLE_SPLIT}_beam{BEAM_WIDTH}_topk{TOP_K}_alpha{ALPHA}.json"
json_path.write_text(json.dumps({
    "split": EXAMPLE_SPLIT, "seed": RANDOM_SEED,
    "beam_width": BEAM_WIDTH, "top_k": TOP_K, "alpha": ALPHA,
    "examples": examples,
}, indent=2), encoding="utf-8")
print("Saved figure:", figure_path)
print("Saved caption JSON:", json_path)

## 11. Define corpus BLEU-1…4 evaluation (five references per image)

This cell uses NLTK's **corpus BLEU** and the same reference tokenisation as your baseline code. All four strategies are evaluated against the **same images and references**.

For a quick check, enable `RUN_EXPLORATORY_BLEU` and use `VALIDATION_IMAGE_LIMIT=20`. Those scores are **not final results**. For your final report, enable `RUN_FINAL_TEST_BLEU` after you have fixed the decoding settings and use the full test set.

**Note:** Random sampling may achieve lower BLEU because BLEU rewards overlap with reference phrasing; it does not directly quantify caption diversity or factual correctness.

In [ ]:
BLEU_WEIGHTS = {
    "BLEU-1": (1., 0., 0., 0.),
    "BLEU-2": (.5, .5, 0., 0.),
    "BLEU-3": (1/3, 1/3, 1/3, 0.),
    "BLEU-4": (.25, .25, .25, .25),
}

def calculate_bleu_for_images(image_ids, split_name):
    references = []
    predictions = {method: [] for method in METHODS}
    duration = {method: [] for method in METHODS}
    rng = np.random.default_rng(RANDOM_SEED)

    for index, img_id in enumerate(image_ids, start=1):
        item = generate_all_methods(img_id, rng)
        references.append([_reference_tokens(c) for c in assets.references[img_id]])
        for method in METHODS:
            predictions[method].append(item["captions"][method].split())
            duration[method].append(item["seconds"][method])
        if index % 100 == 0:
            print(f"{split_name}: captioned {index}/{len(image_ids)} images", flush=True)

    result_rows = []
    for method in METHODS:
        row = {
            "Method": method,
            "Images": len(image_ids),
            "Seconds/caption": float(np.mean(duration[method])),
        }
        for name, weights in BLEU_WEIGHTS.items():
            row[name] = float(corpus_bleu(
                references, predictions[method], weights=weights
            ))
        result_rows.append(row)
    return pd.DataFrame(result_rows)

## 12. Optional: quick validation-set BLEU

Enable `RUN_EXPLORATORY_BLEU=True` if you want to check that the BLEU pipeline runs before evaluating the entire test set. These values can be used for model-selection experiments, but shouldn't be labelled full-test performance.

In [ ]:
if RUN_EXPLORATORY_BLEU:
    validation_ids = assets.splits["val"][:VALIDATION_IMAGE_LIMIT]
    validation_scores = calculate_bleu_for_images(validation_ids, "validation")
    display(validation_scores)
    validation_scores.to_csv(OUTPUT_DIR / "bleu_exploratory_validation.csv", index=False)
else:
    print("Exploratory BLEU skipped (RUN_EXPLORATORY_BLEU=False).")

## 13. Optional: final 810-image test-set BLEU comparison

Run only after parameters `BEAM_WIDTH`, `TOP_K` and `ALPHA` are fixed. Do not tune on this set.

This will perform repeated next-word predictions for all four strategies and may take substantial time. Results are saved as CSV and JSON so a report chart can be regenerated from the *actual* scores, rather than hardcoded example values.

In [ ]:
if RUN_FINAL_TEST_BLEU:
    test_ids = assets.splits["test"]
    full_scores = calculate_bleu_for_images(test_ids, "test")
    display(full_scores)
    full_scores.to_csv(OUTPUT_DIR / "bleu_full_test.csv", index=False)
    summary = {
        "split": "test", "images": len(test_ids),
        "model_dir": str(MODEL_DIR), "features_dir": str(FEATURES_DIR),
        "seed": RANDOM_SEED, "beam_width": BEAM_WIDTH,
        "top_k": TOP_K, "alpha": ALPHA,
        "results": full_scores.to_dict(orient="records"),
    }
    (OUTPUT_DIR / "bleu_full_test.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print("Results saved to:", OUTPUT_DIR)
else:
    print("Final full-test BLEU skipped (RUN_FINAL_TEST_BLEU=False).")

## 14. Graph only the results calculated by this notebook

The chart is built dynamically from your validation or full-test results. It **does not** reuse hardcoded BLEU numbers from PR #7's earlier chart notebook.

In [ ]:
if "full_scores" in globals():
    scores_to_plot = full_scores
    chart_split = "test"
elif "validation_scores" in globals():
    scores_to_plot = validation_scores
    chart_split = "exploratory validation"
else:
    scores_to_plot = None

if scores_to_plot is not None:
    ax = scores_to_plot.set_index("Method")[list(BLEU_WEIGHTS)].T.plot(
        kind="bar", figsize=(10, 5), rot=0
    )
    ax.set_title(f"Flickr8k BLEU comparison — {chart_split}")
    ax.set_ylabel("Corpus BLEU score")
    ax.set_xlabel("Metric")
    ax.grid(axis="y", alpha=0.2)
    ax.figure.tight_layout()
    chart_path = OUTPUT_DIR / f"bleu_{chart_split.replace(' ', '_')}.png"
    ax.figure.savefig(chart_path, dpi=180)
    plt.show()
    plt.close(ax.figure)
    print("Saved chart:", chart_path)
else:
    print("No BLEU metrics yet. Enable one evaluation flag above and rerun.")

## 15. Interpretation and reporting notes

- **Greedy vs standard beam** measures whether broader candidate search helps without retraining the decoder.
- **Length-normalised beam** changes candidate ranking through `alpha`, which may help avoid short captions but does not guarantee better accuracy.
- **Top-k sampling** increases decoding variation. Use a fixed seed for repeatability, and preferably multiple seeds when researching variability.
- **BLEU-4** compares four-token n-gram matches against the reference captions. Higher BLEU does **not** guarantee an image-grounded description; inspect the actual images for hallucinations and missed objects.
- **Validation vs test:** Select parameters using validation results; report test scores once the settings are fixed.
- **Caveat:** The PR #7 implementation uses a simple length normalisation formula with `len(tokens)`, including the start marker. If comparing against another paper's length penalty, document the exact formula and counting convention.

**Saved outputs:** validation examples PNG/JSON; optional validation BLEU CSV; optional final test BLEU CSV/JSON and dynamic chart. No full retraining is needed.